### Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

ROOT = Path.cwd()
if ROOT.name in {"notebook", "notebooks"}:
    ROOT = ROOT.parent

OFFICIAL = ROOT / "datasets" / "raw" / "official_report_csv"
if not OFFICIAL.exists() or not any(OFFICIAL.glob("*.csv")):
    OFFICIAL = ROOT / "datasets" / "raw" / "data" / "raw" / "official_report_csv"
if not OFFICIAL.exists():
    raise FileNotFoundError(f"Official CSV directory not found: {OFFICIAL}")
RESULTS = ROOT / "results" / "tables"

RESULTS.mkdir(parents=True, exist_ok=True)

### Load CEB datasets

In [ ]:
capacity = pd.read_csv(
    OFFICIAL / "ceb_installed_capacity_by_owner_and_type_2024_2025.csv"
)

generation = pd.read_csv(
    OFFICIAL / "ceb_net_generation_by_owner_and_source_2024_2025.csv"
)

demand = pd.read_csv(
    OFFICIAL / "ceb_historical_electricity_demand_2009_2023.csv"
)

expansion = pd.read_csv(
    OFFICIAL / "ceb_base_case_solar_and_storage_additions_2025_2044.csv"
)

### Explore each dataset

In [ ]:
datasets = {
    "Installed Capacity": capacity,
    "Electricity Generation": generation,
    "Historical Demand": demand,
    "Future Expansion": expansion
}

for name, df in datasets.items():

    print("\n", "=" * 50)
    print(name)

    display(df.head(10))

    print("\nInformation:")
    df.info()

    print("\nMissing values:")
    display(df.isnull().sum())

### Export an initial data inventory

In [ ]:
inventory = []

for file in OFFICIAL.glob("*.csv"):

    df = pd.read_csv(file)

    inventory.append({
        "dataset": file.name,
        "records": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum())
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

inventory_df.to_csv(
    RESULTS / "official_data_inventory.csv",
    index=False
)

### Plot installed capacity and generation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
capacity_plot = capacity.assign(
    capacity_mw=pd.to_numeric(capacity["installed_capacity_mw_2025"], errors="coerce")
)
capacity_plot.plot(x="plant_type", y="capacity_mw", kind="bar", ax=axes[0], legend=False)
axes[0].set_title("Installed capacity by plant type (2025)")
axes[0].set_ylabel("Capacity (MW)")
axes[0].tick_params(axis="x", rotation=45)

generation_plot = generation.assign(
    generation_gwh=pd.to_numeric(generation["net_generation_gwh_2025"], errors="coerce")
)
generation_plot.plot(x="generation_source", y="generation_gwh", kind="bar", ax=axes[1], legend=False)
axes[1].set_title("Net generation by source (2025)")
axes[1].set_ylabel("Generation (GWh)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Save selected data

In [ ]:
capacity.to_csv(
    RESULTS / "ceb_capacity_analysis_input.csv",
    index=False
)

generation.to_csv(
    RESULTS / "ceb_generation_analysis_input.csv",
    index=False
)

demand.to_csv(
    RESULTS / "ceb_demand_analysis_input.csv",
    index=False
)

expansion.to_csv(
    RESULTS / "ceb_expansion_analysis_input.csv",
    index=False
)